In [14]:
!cd ..
!git clone https://github.com/ngotruong64/EE5201_DETR_Gr3.git -b develop

Cloning into 'EE5201_DETR_Gr3'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 210 (delta 24), reused 203 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 18.64 MiB | 16.65 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [15]:
%cd EE5201_DETR_Gr3
%cd Defor_DETR_DETopK/
!git branch


/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/EE5201_DETR_Gr3
/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/EE5201_DETR_Gr3/Defor_DETR_DETopK
* develop


In [16]:
!nvidia-smi

Fri Nov 28 02:27:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Path tới folder DETR trong Colab
detr_path = "/content/EE5201_DETR_Gr3/Defor_DETR_DETopK"
import os
# Tạo thư mục data nếu chưa có
data_path = os.path.join(detr_path, "data")
os.makedirs(data_path, exist_ok=True)

print("Data directory:", data_path)

# Đường dẫn file zip trên Drive
zip_path = "/content/drive/MyDrive/00_MASTER/01_HK1/coco.zip"

print("Zip file:", zip_path)

# Giải nén
!unzip -o "$zip_path" -d "$data_path"

print("✅ Giải nén thành công vào:", data_path)

Streaming output truncated to the last 5000 lines.
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005494_jpg.rf.695eafd03e6ea75a5ad6c8b8ca830938.jpg  
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005512_jpg.rf.b6023800c60bed07e7c5704b21e6a463.jpg  
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005512_jpg.rf.bc557e009d0540ec38ea44a62180c1de.jpg  
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005512_jpg.rf.c60967c318454794aefa403f688f8413.jpg  
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005515_jpg.rf.57111bf23e43f2e3725a5b530af4b0d2.jpg  
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005515_jpg.rf.7651507057a6f4de76e0ffaa0688036b.jpg  
  inflating: /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/data/coco/train2017/2010_005515_jpg.rf.e7cdc9d1b89209ee86f8dcd0f977e352.jpg  
  inflating

In [19]:
!pip install -r requirements.txt
!pip install ninja

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/MultiScaleDeformableAttention-1.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/MultiScaleDeformableAttention-1.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [20]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/MultiScaleDeformableAttention-1.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://download.pytorch.org/whl/cu124


In [21]:
!pip install setuptools
!ls
%cd models/ops

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/MultiScaleDeformableAttention-1.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
benchmark.py  configs	docs	   figs     main.py  README.md	       tools
code.txt      datasets	engine.py  LICENSE  models   requirements.txt  util
/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/EE5201_DETR_Gr3/Defor_DETR_DETopK/models/ops


In [22]:
%cd /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/models/ops/src/cuda
!perl -pi -e 's/value\.scalar_type\(\)\.is_cuda\(\)/value.is_cuda()/g' ms_deform_attn_cuda.cu
!perl -pi -e 's/AT_DISPATCH_FLOATING_TYPES\(value.type\(\)/AT_DISPATCH_FLOATING_TYPES(value.scalar_type()/g' ms_deform_attn_cuda.cu


/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/models/ops/src/cuda


In [23]:
%cd /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/models/ops
!python setup.py build install

/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/models/ops
running build
running build_py
copying functions/__init__.py -> build/lib.linux-x86_64-cpython-312/functions
copying functions/ms_deform_attn_func.py -> build/lib.linux-x86_64-cpython-312/functions
copying modules/__init__.py -> build/lib.linux-x86_64-cpython-312/modules
copying modules/ms_deform_attn.py -> build/lib.linux-x86_64-cpython-312/modules
running build_ext
W1128 02:28:14.745000 20910 torch/utils/cpp_extension.py:521] The detected CUDA version (12.5) has a minor version mismatch with the version that was used to compile PyTorch (12.6). Most likely this shouldn't be a problem.
W1128 02:28:14.746000 20910 torch/utils/cpp_extension.py:531] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.5
building 'MultiScaleDeformableAttention' extension
[1/1] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/models/ops/build/temp.linux-x8

In [24]:
%cd /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/
!python main.py --top_K_matching --set_top_K=5 --top_K_matching_method='closest' --top_K_scale=0.5 --num_feature_levels=4 --batch_size 2 --lr 6.25e-6 --coco_path data/coco --resume /content/drive/MyDrive/00_MASTER/01_HK1/output/checkpoint41.pth   --output_dir /content/drive/MyDrive/00_MASTER/01_HK1/output

/content/EE5201_DETR_Gr3/Defor_DETR_DETopK
Not using distributed mode
git:
  sha: c53558c36209dc49d90b2aa4fa8b13e568d19547, status: has uncommited changes, branch: develop

Namespace(lr=6.25e-06, lr_backbone_names=['backbone.0'], lr_backbone=2e-05, lr_linear_proj_names=['reference_points', 'sampling_offsets'], lr_linear_proj_mult=0.1, batch_size=2, weight_decay=0.0001, epochs=50, lr_drop=40, lr_drop_epochs=None, clip_max_norm=0.1, sgd=False, with_box_refine=False, two_stage=False, frozen_weights=None, backbone='resnet50', dilation=False, position_embedding='sine', position_embedding_scale=6.283185307179586, num_feature_levels=4, enc_layers=6, dec_layers=6, dim_feedforward=1024, hidden_dim=256, dropout=0.1, nheads=8, num_queries=300, dec_n_points=4, enc_n_points=4, masks=False, aux_loss=True, set_cost_class=2, set_cost_bbox=5, set_cost_giou=2, top_K_matching=True, top_K_matching_method='closest', set_top_K=5, top_K_scale=0.5, mask_loss_coef=1, dice_loss_coef=1, cls_loss_coef=2, bbox_los

In [25]:
%cd /content/EE5201_DETR_Gr3/Defor_DETR_DETopK/
!python benchmark.py

/content/EE5201_DETR_Gr3/Defor_DETR_DETopK
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Traceback (most recent call last):
  File "/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/benchmark.py", line 65, in <module>
    fps = benchmark()
          ^^^^^^^^^^^
  File "/content/EE5201_DETR_Gr3/Defor_DETR_DETopK/benchmark.py", line 53, in benchma

In [42]:
!python visualize_detections.py \
  --resume /content/drive/MyDrive/00_MASTER/01_HK1/output/checkpoint0049.pth \
  --split val \
  --num_images 15 \
  --score_thresh 0.2 \
  --save_dir vis_epoch49 \
  --coco_path data/coco \
  --batch_size 1 \
  --top_K_matching --set_top_K=5 --top_K_matching_method='closest' \
  --top_K_scale=0.5 --num_feature_levels=4

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loaded checkpoint from /content/drive/MyDrive/00_MASTER/01_HK1/output/checkpoint0049.pth
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/at

In [46]:
!python main.py \
  --eval \
  --resume /content/drive/MyDrive/00_MASTER/01_HK1/output/checkpoint0049.pth \
  --dataset_file coco \
  --coco_path data/coco \
  --batch_size 2 \
  --num_workers 2 \
  --top_K_matching --set_top_K=5 \
  --top_K_matching_method closest \
  --top_K_scale 0.5 \
  --num_feature_levels 4


Not using distributed mode
git:
  sha: bcd61286b1d9aeed9dbbdfcdf81ac9460e5f55bc, status: has uncommited changes, branch: develop

Namespace(lr=0.0002, lr_backbone_names=['backbone.0'], lr_backbone=2e-05, lr_linear_proj_names=['reference_points', 'sampling_offsets'], lr_linear_proj_mult=0.1, batch_size=2, weight_decay=0.0001, epochs=50, lr_drop=40, lr_drop_epochs=None, clip_max_norm=0.1, sgd=False, with_box_refine=False, two_stage=False, frozen_weights=None, backbone='resnet50', dilation=False, position_embedding='sine', position_embedding_scale=6.283185307179586, num_feature_levels=4, enc_layers=6, dec_layers=6, dim_feedforward=1024, hidden_dim=256, dropout=0.1, nheads=8, num_queries=300, dec_n_points=4, enc_n_points=4, masks=False, aux_loss=True, set_cost_class=2, set_cost_bbox=5, set_cost_giou=2, top_K_matching=True, top_K_matching_method='closest', set_top_K=5, top_K_scale=0.5, mask_loss_coef=1, dice_loss_coef=1, cls_loss_coef=2, bbox_loss_coef=5, giou_loss_coef=2, K_cls_loss_coef=0